<a href="https://colab.research.google.com/github/mrhelek/proyekakhirNLP/blob/main/skenario_1_baseline(dataset_dan_LLM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SKENARIO 1: EVALUASI BASELINE (TinyLlama Full Precision)
# ============================================================
# Cell ini akan:
# 1. Clone repo BitDistiller (jika belum ada)
# 2. Install dependensi (transformers 4.37.0, accelerate 0.28.0, dsb)
# 3. Download model TinyLlama 1.1B (full precision)
# 4. Jalankan evaluasi perplexity (WikiText-2) dan zero-shot (HellaSwag)
# 5. Simpan hasil dalam JSON dan tampilkan tabel perbandingan
# ============================================================

import os
import subprocess
import re
import json
import pandas as pd
from huggingface_hub import snapshot_download

# ---------- 1. Setup environment ----------
print("🔧 Menyiapkan environment (clone repo & install dependensi)...")
if not os.path.exists("/content/BitDistiller"):
    !git clone https://github.com/BrownianNotion/BitDistiller.git
%cd /content/BitDistiller
!pip install -r requirements.txt
!pip install protobuf sentencepiece

# ---------- 2. Download model baseline ----------
print("\n📥 Mendownload model baseline: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T ...")
local_model_dir = "/content/models/tinyllama_base"
os.makedirs(local_model_dir, exist_ok=True)
snapshot_download(
    repo_id="TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    local_dir=local_model_dir,
    ignore_patterns=["*.bin", "*.h5"]  # hanya download .safetensors
)
print("✅ Model baseline siap.\n")

# ---------- 3. Evaluasi ----------
eval_dir = "/content/BitDistiller/test/general"
os.chdir(eval_dir)

# Gunakan path absolut agar script dapat menemukan model
model_abs_path = os.path.abspath(local_model_dir)
print(f"📂 Model absolute path: {model_abs_path}\n")

print("=" * 60)
print("📊 [1/2] Menghitung WikiText-2 Perplexity (full precision)...")
print("=" * 60)
cmd_ppl = [
    "python", "wiki_ppl.py",
    "--model", model_abs_path,
    "--quant_type", "fp",
    "--bits", "16",
    "--group_size", "128"
]
proc_ppl = subprocess.run(cmd_ppl, capture_output=True, text=True)
print(proc_ppl.stdout)
if proc_ppl.stderr:
    print("STDERR:", proc_ppl.stderr)

ppl_match = re.search(r"ppl:\s*\n?\s*([\d.]+)", proc_ppl.stdout + proc_ppl.stderr)
if ppl_match:
    ppl_base = round(float(ppl_match.group(1)), 4)
    print(f"\n✅ Perplexity (WikiText-2): {ppl_base}")
else:
    ppl_base = None
    print("\n⚠️ Gagal mengekstrak PPL. Periksa output di atas.")

print("\n" + "=" * 60)
print("📊 [2/2] Menghitung Zero-shot Accuracy (HellaSwag)...")
print("=" * 60)
cmd_qa = [
    "python", "llm_eval.py",
    "--model", model_abs_path,
    "--eval_tasks", "hellaswag",
    "--test_set",
    "--bits", "16",
    "--group_size", "128",
    "--quant_type", "fp",
    "--num_fewshot", "0",
    "--batch_size", "2"
]
proc_qa = subprocess.run(cmd_qa, capture_output=True, text=True)
print(proc_qa.stdout)
if proc_qa.stderr:
    print("STDERR:", proc_qa.stderr)

# Cari acc_norm (prioritas) atau acc biasa
acc_match = re.search(r"'acc_norm':\s*([\d.]+)", proc_qa.stdout + proc_qa.stderr)
if not acc_match:
    acc_match = re.search(r"'acc':\s*([\d.]+)", proc_qa.stdout + proc_qa.stderr)
if acc_match:
    acc_base = round(float(acc_match.group(1)) * 100, 2)
    print(f"\n✅ HellaSwag acc_norm: {acc_base}%")
else:
    acc_base = None
    print("\n⚠️ Gagal mengekstrak akurasi. Periksa output di atas.")

# ---------- 4. Simpan hasil ----------
results = {
    "model": "TinyLlama-1.1B (full precision)",
    "wikitext2_ppl": ppl_base,
    "hellaswag_acc_norm": acc_base
}
os.makedirs("/content/results", exist_ok=True)
with open("/content/results/scenario1_baseline.json", "w") as f:
    json.dump(results, f, indent=2)

df = pd.DataFrame([results]).T
df.columns = ["Value"]
print("\n" + "=" * 60)
print("📋 HASIL AKHIR SKENARIO 1 (BASELINE)")
print(df.to_markdown())
print("=" * 60)
print("💾 Hasil disimpan di /content/results/scenario1_baseline.json")

# Tampilkan perbandingan cepat dengan BitDistiller 2-bit (dari hasil sebelumnya)
print("\n📌 Perbandingan dengan BitDistiller 2-bit (hasil yang sudah Anda peroleh):")
print(f"   - Baseline PPL       : {ppl_base}")
print(f"   - BitDistiller 2-bit PPL : 23.7603")
print(f"   - Baseline HellaSwag : {acc_base}%")
print(f"   - BitDistiller 2-bit HellaSwag : 37.78%")

🔧 Menyiapkan environment (clone repo & install dependensi)...
/content/BitDistiller

📥 Mendownload model baseline: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Output streaming akan dipotong hingga 5000 baris terakhir.
100%|██████████| 40145/40145 [11:22<00:00, 58.79it/s]


✅ HellaSwag acc_norm: 59.16%

📋 HASIL AKHIR SKENARIO 1 (BASELINE)
|                    | Value                           |
|:-------------------|:--------------------------------|
| model              | TinyLlama-1.1B (full precision) |
| wikitext2_ppl      | 7.7747                          |
| hellaswag_acc_norm | 59.16                           |
💾 Hasil disimpan di /content/results/scenario1_baseline.json

📌 Perbandingan dengan BitDistiller 2-bit (hasil yang sudah Anda peroleh):
   - Baseline PPL       : 7.7747
   - BitDistiller 2-bit PPL : 23.7603
   - Baseline HellaSwag : 59.16%
   - BitDistiller 2-bit HellaSwag : 37.78%
